# MathLive + robust identifier system showcase

This notebook is intentionally **figure-free**. It only exercises the semantic MathLive input layer, the canonical identifier system, and the MathJSON transport used between MathLive and the toolkit backend.


In [ ]:
%pip -q install sympy anywidget ipywidgets
import import_setup


In [ ]:
import sympy as sp
from IPython.display import display

from gu_toolkit import ExpressionContext, ExpressionInput, IdentifierInput, NamedFunction, parse_identifier, render_latex, symbol


In [ ]:
@NamedFunction
def Force(x):
    return x

@NamedFunction
def Force__x(x):
    return x


In [ ]:
identifier_examples = {
    'velocity': r'\mathrm{velocity}',
    'theta__x': r'\mathrm{theta\_x}',
    'a_1_2': r'a_{1,2}',
}
for canonical, latex in identifier_examples.items():
    assert parse_identifier(canonical) == canonical
    assert parse_identifier(latex) == canonical

function_examples = {
    'Force': r'\operatorname{Force}',
    'Force_t': r'\operatorname{Force}_{t}',
    'Force__x': r'\operatorname{Force\_x}',
}
for canonical, latex in function_examples.items():
    assert parse_identifier(latex) == canonical

print('Identifier grammar checks passed.')


In [ ]:
velocity = symbol('velocity')
theta_x = symbol('theta__x')
a_12 = symbol('a_1_2')
x = symbol('x')
ctx = ExpressionContext.from_symbols(
    [velocity, theta_x, a_12, x],
    functions=[Force, Force__x, 'Force_t'],
    include_named_functions=False,
)
manifest = ctx.transport_manifest(field_role='expression')
symbols_by_name = {entry['name']: entry for entry in manifest['symbols']}
functions_by_name = {entry['name']: entry for entry in manifest['functions']}
assert symbols_by_name['velocity']['latex'] == r'\mathrm{velocity}'
assert symbols_by_name['theta__x']['latex'] == r'\mathrm{theta\_x}'
assert functions_by_name['Force']['latexHead'] == r'\operatorname{Force}'
assert functions_by_name['Force__x']['latexHead'] == r'\operatorname{Force\_x}'
assert functions_by_name['Force_t']['latexHead'] == r'\operatorname{Force}_{t}'
manifest


In [ ]:
identifier_widget = IdentifierInput(context=ctx, value=r'\mathrm{theta\_x}')
expression_widget = ExpressionInput(
    context=ctx,
    value=r'\operatorname{Force}(x) + \mathrm{velocity}',
)
display(identifier_widget)
display(expression_widget)

assert identifier_widget.parse_value() == 'theta__x'
assert expression_widget.parse_value() == Force(x) + velocity
print('Rendered widgets above are pure math-input widgets; no Figure objects are involved.')


In [ ]:
identifier_widget.value = ''
identifier_widget.math_json = ['Subscript', 'a', ['Tuple', 1, 2]]
assert identifier_widget.parse_value() == 'a_1_2'

expression_widget.value = ''
expression_widget.math_json = [
    'Add',
    ['Force', 'theta__x'],
    ['Multiply', 'velocity', 'x'],
]
parsed = expression_widget.parse_value()
assert parsed == Force(theta_x) + velocity * x
latex = ctx.render_latex(parsed)
assert r'\operatorname{Force}' in latex
assert r'\mathrm{velocity}' in latex
print('MathJSON transport checks passed.')


## Interactive checks

Try typing names such as `velocity`, `theta__x`, `a_1_2`, `Force`, `Force_t`, and `Force__x` into the widgets above. The backend transport should keep variable names canonical, render functions with `\operatorname{...}`, and preserve context-aware atomic parsing through the MathJSON bridge.
